In [2]:
# Cell 1: Load enriched data from Phase 1
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
PROCESSED = PROJECT_ROOT / "data" / "processed"

df = pd.read_csv(PROCESSED / "transactions_enriched.csv")
df['date_dt'] = pd.to_datetime(df['date_iso'])

print(f"Loaded: {df.shape[0]} transactions, {df.shape[1]} columns")
print(f"Period: {df['date_iso'].min()} to {df['date_iso'].max()}")

Loaded: 193 transactions, 15 columns
Period: 2026-01-02 to 2026-03-09


In [3]:
# Cell 2: Spending Anomaly Detection
# WHY: Users want to know "what was that big charge?" without
# scrolling through hundreds of transactions. We flag anything
# that is significantly higher than normal for that category.
#
# METHOD: For each category, calculate the mean and standard
# deviation of transaction amounts. Anything more than 2 standard
# deviations above the mean is flagged as unusual.

spending = df[df['direction'] == 'OUT'].copy()

anomalies = []

for category in spending['category'].unique():
    cat_txns = spending[spending['category'] == category]
    
    if len(cat_txns) < 3:
        # Not enough data to determine what is "normal"
        # Flag any single transaction over GBP 100 as notable
        for _, row in cat_txns.iterrows():
            if row['money_out'] > 100:
                anomalies.append({
                    'date': row['date_iso'],
                    'merchant': row['merchant'],
                    'category': category,
                    'amount': row['money_out'],
                    'reason': f"Large one-off ({category})",
                })
        continue
    
    mean = cat_txns['money_out'].mean()
    std = cat_txns['money_out'].std()
    threshold = mean + (2 * std)
    
    for _, row in cat_txns.iterrows():
        if row['money_out'] > threshold and row['money_out'] > 10:
            anomalies.append({
                'date': row['date_iso'],
                'merchant': row['merchant'],
                'category': category,
                'amount': row['money_out'],
                'avg_for_category': round(mean, 2),
                'reason': f"GBP {row['money_out']:.2f} vs avg GBP {mean:.2f} for {category}",
            })

anomalies_df = pd.DataFrame(anomalies).sort_values('amount', ascending=False)

print(f"=== SPENDING ANOMALIES DETECTED: {len(anomalies_df)} ===\n")
for _, row in anomalies_df.iterrows():
    print(f"  {row['date']}  {row['merchant']:<30} GBP {row['amount']:>8,.2f}")
    print(f"             {row['reason']}")
    print()

=== SPENDING ANOMALIES DETECTED: 11 ===

  2026-03-02  Remitly (Intl Transfer)        GBP   665.00
             GBP 665.00 vs avg GBP 93.04 for Transfers

  2026-01-07  WACV Conference                GBP   521.96
             Large one-off (Professional)

  2026-02-17  [REDACTED]                     GBP   450.00
             GBP 450.00 vs avg GBP 93.04 for Transfers

  2026-01-27  Information Services           GBP   149.85
             Large one-off (Bills)

  2026-02-20  Sainsburys                     GBP    47.86
             GBP 47.86 vs avg GBP 6.28 for Groceries

  2026-02-02  Shein                          GBP    43.35
             GBP 43.35 vs avg GBP 17.93 for Shopping

  2026-02-19  Southampton Harbour Hotel      GBP    38.06
             GBP 38.06 vs avg GBP 7.93 for Eating Out

  2026-01-16  International Food Store       GBP    27.66
             GBP 27.66 vs avg GBP 6.28 for Groceries

  2026-02-19  Uber                           GBP    25.25
             GBP 25.25 vs avg

In [4]:
# Cell 3: Month-over-Month Comparison
# WHY: Users want to know if their spending is going up or down.
# "You spent 40% more on eating out in February vs January" is
# the kind of insight that drives behaviour change.

spending = df[df['direction'] == 'OUT'].copy()
spending['month'] = pd.to_datetime(spending['date_iso']).dt.to_period('M')

# Only compare Jan vs Feb (March is incomplete)
jan = spending[spending['month'] == '2026-01']
feb = spending[spending['month'] == '2026-02']

jan_by_cat = jan.groupby('category')['money_out'].sum()
feb_by_cat = feb.groupby('category')['money_out'].sum()

# Combine into one table
comparison = pd.DataFrame({
    'January': jan_by_cat,
    'February': feb_by_cat
}).fillna(0)

comparison['Change'] = comparison['February'] - comparison['January']
comparison['Change %'] = ((comparison['February'] - comparison['January']) / comparison['January'].replace(0, np.nan) * 100).round(1)
comparison = comparison.sort_values('Change', ascending=False)

print("=== JANUARY vs FEBRUARY SPENDING ===\n")
print(f"  {'Category':<25} {'January':>10} {'February':>10} {'Change':>10} {'%':>8}")
print(f"  {'-'*65}")

for cat, row in comparison.iterrows():
    arrow = '▲' if row['Change'] > 0 else ('▼' if row['Change'] < 0 else ' ')
    pct = f"{row['Change %']:.0f}%" if not pd.isna(row['Change %']) else 'NEW'
    print(f"  {cat:<25} GBP {row['January']:>7,.2f} GBP {row['February']:>7,.2f}  {arrow} GBP {abs(row['Change']):>6,.2f}  {pct:>6}")

jan_total = comparison['January'].sum()
feb_total = comparison['February'].sum()
change = feb_total - jan_total
print(f"  {'-'*65}")
print(f"  {'TOTAL':<25} GBP {jan_total:>7,.2f} GBP {feb_total:>7,.2f}  {'▲' if change > 0 else '▼'} GBP {abs(change):>6,.2f}  {change/jan_total*100:.0f}%")

=== JANUARY vs FEBRUARY SPENDING ===

  Category                     January   February     Change        %
  -----------------------------------------------------------------
  Transfers                 GBP  395.07 GBP 1,024.92  ▲ GBP 629.85    159%
  Cash                      GBP    0.00 GBP   80.00  ▲ GBP  80.00     NEW
  Food Delivery             GBP    0.00 GBP   38.20  ▲ GBP  38.20     NEW
  Eating Out                GBP   65.27 GBP  101.25  ▲ GBP  35.98     55%
  Subscriptions             GBP    5.72 GBP   17.67  ▲ GBP  11.95    209%
  Coffee & Cafe             GBP    0.00 GBP   10.60  ▲ GBP  10.60     NEW
  Shopping                  GBP   68.11 GBP   59.35  ▼ GBP   8.76    -13%
  Bank Fees                 GBP   17.17 GBP    2.86  ▼ GBP  14.31    -83%
  Groceries                 GBP  107.90 GBP   74.22  ▼ GBP  33.68    -31%
  Transport                 GBP  204.51 GBP  162.90  ▼ GBP  41.61    -20%
  Rent                      GBP  780.00 GBP  660.00  ▼ GBP 120.00    -15%
  Bills  

In [5]:
# Cell 4: Synthetic Data Generator
# WHY: We need test data for multiple banks and larger datasets.
# This generator creates realistic transaction data that mimics
# real UK and international banking patterns.
#
# The output is in our STANDARD CSV format -- the same format
# that any bank parser would produce. This proves that the
# intelligence layer works regardless of the bank source.

import random
from datetime import datetime, timedelta

# Realistic merchant profiles for synthetic data
# Each has a name, category, typical amount range, and frequency
MERCHANT_PROFILES = [
    # Rent / Housing
    {'name': 'Landlord Payment', 'category': 'Rent', 'min': 600, 'max': 1200, 'monthly': True},
    {'name': 'Council Tax', 'category': 'Bills', 'min': 100, 'max': 200, 'monthly': True},
    {'name': 'British Gas', 'category': 'Bills', 'min': 40, 'max': 120, 'monthly': True},
    {'name': 'Thames Water', 'category': 'Bills', 'min': 25, 'max': 50, 'monthly': True},
    
    # Subscriptions
    {'name': 'Netflix', 'category': 'Subscriptions', 'min': 10.99, 'max': 17.99, 'monthly': True},
    {'name': 'Spotify', 'category': 'Subscriptions', 'min': 10.99, 'max': 10.99, 'monthly': True},
    {'name': 'Amazon Prime', 'category': 'Subscriptions', 'min': 8.99, 'max': 8.99, 'monthly': True},
    {'name': 'Gym Membership', 'category': 'Subscriptions', 'min': 20, 'max': 45, 'monthly': True},
    {'name': 'iCloud Storage', 'category': 'Subscriptions', 'min': 0.99, 'max': 2.99, 'monthly': True},
    
    # Groceries
    {'name': 'Tesco', 'category': 'Groceries', 'min': 5, 'max': 80, 'monthly': False, 'freq': 8},
    {'name': 'Sainsburys', 'category': 'Groceries', 'min': 5, 'max': 60, 'monthly': False, 'freq': 4},
    {'name': 'Aldi', 'category': 'Groceries', 'min': 10, 'max': 50, 'monthly': False, 'freq': 6},
    {'name': 'Lidl', 'category': 'Groceries', 'min': 5, 'max': 40, 'monthly': False, 'freq': 3},
    
    # Transport
    {'name': 'TfL', 'category': 'Transport', 'min': 2, 'max': 8, 'monthly': False, 'freq': 20},
    {'name': 'Uber', 'category': 'Transport', 'min': 5, 'max': 25, 'monthly': False, 'freq': 6},
    {'name': 'Trainline', 'category': 'Transport', 'min': 10, 'max': 80, 'monthly': False, 'freq': 2},
    {'name': 'Shell Fuel', 'category': 'Transport', 'min': 30, 'max': 70, 'monthly': False, 'freq': 3},
    
    # Eating Out
    {'name': 'Nandos', 'category': 'Eating Out', 'min': 8, 'max': 25, 'monthly': False, 'freq': 2},
    {'name': 'McDonalds', 'category': 'Eating Out', 'min': 3, 'max': 12, 'monthly': False, 'freq': 4},
    {'name': 'Greggs', 'category': 'Eating Out', 'min': 2, 'max': 8, 'monthly': False, 'freq': 6},
    {'name': 'Pizza Express', 'category': 'Eating Out', 'min': 15, 'max': 40, 'monthly': False, 'freq': 1},
    
    # Food Delivery
    {'name': 'Deliveroo', 'category': 'Food Delivery', 'min': 12, 'max': 35, 'monthly': False, 'freq': 4},
    {'name': 'Uber Eats', 'category': 'Food Delivery', 'min': 10, 'max': 30, 'monthly': False, 'freq': 3},
    {'name': 'Just Eat', 'category': 'Food Delivery', 'min': 10, 'max': 28, 'monthly': False, 'freq': 2},
    
    # Shopping
    {'name': 'Amazon', 'category': 'Shopping', 'min': 5, 'max': 100, 'monthly': False, 'freq': 4},
    {'name': 'Primark', 'category': 'Shopping', 'min': 10, 'max': 50, 'monthly': False, 'freq': 1},
    {'name': 'ASOS', 'category': 'Shopping', 'min': 15, 'max': 80, 'monthly': False, 'freq': 1},
    {'name': 'Boots', 'category': 'Shopping', 'min': 5, 'max': 30, 'monthly': False, 'freq': 2},
    
    # Coffee
    {'name': 'Starbucks', 'category': 'Coffee & Cafe', 'min': 3, 'max': 6, 'monthly': False, 'freq': 8},
    {'name': 'Costa Coffee', 'category': 'Coffee & Cafe', 'min': 3, 'max': 5, 'monthly': False, 'freq': 5},
    
    # Cash
    {'name': 'ATM Withdrawal', 'category': 'Cash', 'min': 20, 'max': 100, 'monthly': False, 'freq': 2},
]

# Income profiles
INCOME_PROFILES = [
    {'name': 'Employer Salary', 'amount_range': (1800, 3500), 'day_of_month': (25, 30)},
    {'name': 'Side Gig Payment', 'amount_range': (50, 300), 'probability': 0.3},
]


def generate_synthetic_data(num_months=6, seed=42):
    """Generate realistic synthetic bank transaction data."""
    random.seed(seed)
    np.random.seed(seed)
    
    transactions = []
    start_date = datetime(2025, 7, 1)
    end_date = start_date + timedelta(days=num_months * 30)
    balance = round(random.uniform(500, 2000), 2)
    
    current_date = start_date
    
    while current_date < end_date:
        month_num = current_date.month
        
        # Generate salary (near end of month)
        if current_date.day == random.randint(25, 28):
            for income in INCOME_PROFILES:
                if 'day_of_month' in income:
                    amount = round(random.uniform(*income['amount_range']), 2)
                    balance += amount
                    transactions.append({
                        'date': current_date.strftime('%d %b %y'),
                        'date_iso': current_date.strftime('%Y-%m-%d'),
                        'description': income['name'],
                        'merchant': income['name'],
                        'type': 'FPI',
                        'money_in': amount,
                        'money_out': '',
                        'balance': round(balance, 2),
                        'direction': 'IN',
                        'category': 'Income',
                    })
                elif random.random() < income.get('probability', 0):
                    amount = round(random.uniform(*income['amount_range']), 2)
                    balance += amount
                    transactions.append({
                        'date': current_date.strftime('%d %b %y'),
                        'date_iso': current_date.strftime('%Y-%m-%d'),
                        'description': income['name'],
                        'merchant': income['name'],
                        'type': 'FPI',
                        'money_in': amount,
                        'money_out': '',
                        'balance': round(balance, 2),
                        'direction': 'IN',
                        'category': 'Income',
                    })
        
        # Generate spending for this day
        for merchant in MERCHANT_PROFILES:
            should_transact = False
            
            if merchant.get('monthly'):
                # Monthly charges happen on a fixed day (1st-5th)
                if current_date.day == random.randint(1, 5):
                    should_transact = True
            else:
                # Random frequency: freq times per month = freq/30 chance per day
                daily_prob = merchant.get('freq', 1) / 30
                if random.random() < daily_prob:
                    should_transact = True
            
            if should_transact:
                amount = round(random.uniform(merchant['min'], merchant['max']), 2)
                balance -= amount
                transactions.append({
                    'date': current_date.strftime('%d %b %y'),
                    'date_iso': current_date.strftime('%Y-%m-%d'),
                    'description': merchant['name'],
                    'merchant': merchant['name'],
                    'type': 'DEB',
                    'money_in': '',
                    'money_out': amount,
                    'balance': round(balance, 2),
                    'direction': 'OUT',
                    'category': merchant['category'],
                })
        
        current_date += timedelta(days=1)
    
    return pd.DataFrame(transactions)

# Generate 6 months of synthetic data
synthetic_df = generate_synthetic_data(num_months=6)

# Save it
synth_path = PROJECT_ROOT / "data" / "synthetic" / "demo_transactions.csv"
synthetic_df.to_csv(synth_path, index=False)

print(f"Synthetic dataset generated!")
print(f"  Transactions: {len(synthetic_df)}")
print(f"  Period: {synthetic_df['date_iso'].min()} to {synthetic_df['date_iso'].max()}")
print(f"  Unique merchants: {synthetic_df['merchant'].nunique()}")
print(f"  Categories: {synthetic_df['category'].nunique()}")
print(f"\n  Spending breakdown:")

synth_spending = synthetic_df[synthetic_df['direction'] == 'OUT']
for cat, total in synth_spending.groupby('category')['money_out'].sum().sort_values(ascending=False).items():
    print(f"    {cat:<20} GBP {total:>8,.2f}")

print(f"\n  Saved to: {synth_path}")

Synthetic dataset generated!
  Transactions: 642
  Period: 2025-07-01 to 2025-12-27
  Unique merchants: 33
  Categories: 11

  Spending breakdown:
    Groceries            GBP 4,339.74
    Rent                 GBP 2,721.78
    Transport            GBP 2,525.42
    Shopping             GBP 1,713.67
    Bills                GBP 1,462.26
    Food Delivery        GBP 1,208.91
    Eating Out           GBP   909.91
    Subscriptions        GBP   488.54
    Cash                 GBP   371.60
    Coffee & Cafe        GBP   346.17

  Saved to: C:\Users\riyaw\OneDrive\Desktop\SpendScope\data\synthetic\demo_transactions.csv


In [6]:
# Cell 5: Prove intelligence layer works on synthetic data
# WHY: If the same anomaly detector and month-over-month code
# works on both real Lloyds data and synthetic data, it proves
# the engine is bank-agnostic. That is a product, not a demo.

synth_spending = synthetic_df[synthetic_df['direction'] == 'OUT'].copy()

# Run anomaly detection on synthetic data
anomalies = []
for category in synth_spending['category'].unique():
    cat_txns = synth_spending[synth_spending['category'] == category]
    if len(cat_txns) < 3:
        continue
    mean = cat_txns['money_out'].mean()
    std = cat_txns['money_out'].std()
    threshold = mean + (2 * std)
    
    for _, row in cat_txns.iterrows():
        if row['money_out'] > threshold and row['money_out'] > 10:
            anomalies.append({
                'date': row['date_iso'],
                'merchant': row['merchant'],
                'amount': row['money_out'],
                'reason': f"GBP {row['money_out']:.2f} vs avg GBP {mean:.2f} for {category}",
            })

print(f"=== ANOMALIES IN SYNTHETIC DATA: {len(anomalies)} ===\n")
for a in sorted(anomalies, key=lambda x: -x['amount'])[:8]:
    print(f"  {a['date']}  {a['merchant']:<20} GBP {a['amount']:>7,.2f}")
    print(f"             {a['reason']}")
    print()

# Subscription detection on synthetic data
synth_spending['month'] = pd.to_datetime(synth_spending['date_iso']).dt.to_period('M')
merchant_monthly = synth_spending.groupby('merchant').agg(
    months_active=('month', 'nunique'),
    total_transactions=('money_out', 'size'),
    avg_amount=('money_out', 'mean'),
    total_spent=('money_out', 'sum'),
).reset_index()

subs = merchant_monthly[(merchant_monthly['months_active'] >= 5) & 
                         (merchant_monthly['total_transactions'] <= merchant_monthly['months_active'] * 2)]

print(f"=== SUBSCRIPTIONS IN SYNTHETIC DATA ===\n")
for _, row in subs.sort_values('total_spent', ascending=False).iterrows():
    monthly = row['total_spent'] / row['months_active']
    print(f"  {row['merchant']:<20} GBP {monthly:>7,.2f}/month  ({int(row['months_active'])} months)")

print(f"\n  Same code. Different data. Bank-agnostic engine confirmed.")

=== ANOMALIES IN SYNTHETIC DATA: 30 ===

  2025-11-23  Amazon               GBP   88.86
             GBP 88.86 vs avg GBP 32.96 for Shopping

  2025-09-01  Amazon               GBP   82.25
             GBP 82.25 vs avg GBP 32.96 for Shopping

  2025-08-10  Tesco                GBP   79.59
             GBP 79.59 vs avg GBP 35.00 for Groceries

  2025-09-24  Trainline            GBP   79.41
             GBP 79.41 vs avg GBP 13.80 for Transport

  2025-11-27  Trainline            GBP   78.85
             GBP 78.85 vs avg GBP 13.80 for Transport

  2025-11-24  Amazon               GBP   78.01
             GBP 78.01 vs avg GBP 32.96 for Shopping

  2025-10-12  Tesco                GBP   77.76
             GBP 77.76 vs avg GBP 35.00 for Groceries

  2025-09-07  Tesco                GBP   77.63
             GBP 77.63 vs avg GBP 35.00 for Groceries

=== SUBSCRIPTIONS IN SYNTHETIC DATA ===

  Council Tax          GBP  180.16/month  (5 months)
  Trainline            GBP   97.09/month  (6 months)

In [7]:
# Cell 6: Savings Opportunity Calculator
# WHY: Showing someone "you spend GBP 47 on food delivery" is 
# informative. Telling them "cutting food delivery in half would
# save you GBP 282 per year" is actionable. That difference is
# what makes a product useful vs just interesting.

# Use the real Lloyds data for this
spending = df[df['direction'] == 'OUT'].copy()
spending['month'] = pd.to_datetime(spending['date_iso']).dt.to_period('M')

# Calculate monthly averages (using Jan and Feb only, March is incomplete)
full_months = spending[spending['month'].isin([pd.Period('2026-01'), pd.Period('2026-02')])]
months_count = 2

monthly_by_cat = full_months.groupby('category')['money_out'].sum() / months_count

# Define savings opportunities with realistic suggestions
SAVINGS_RULES = [
    {
        'category': 'Food Delivery',
        'suggestion': 'Cook instead of ordering delivery once a week',
        'save_pct': 0.50,
    },
    {
        'category': 'Eating Out',
        'suggestion': 'Meal prep lunches instead of eating out twice a week',
        'save_pct': 0.30,
    },
    {
        'category': 'Coffee & Cafe',
        'suggestion': 'Make coffee at home 3 days a week',
        'save_pct': 0.60,
    },
    {
        'category': 'Transport',
        'suggestion': 'Walk or cycle for short trips, use bus more',
        'save_pct': 0.20,
    },
    {
        'category': 'Subscriptions',
        'suggestion': 'Review and cancel unused subscriptions',
        'save_pct': 0.30,
    },
    {
        'category': 'Shopping',
        'suggestion': 'Implement a 48-hour rule before non-essential purchases',
        'save_pct': 0.25,
    },
    {
        'category': 'Cash',
        'suggestion': 'Track cash spending (currently invisible to the app)',
        'save_pct': 0.30,
    },
    {
        'category': 'Bank Fees',
        'suggestion': 'Use a fee-free card for international purchases',
        'save_pct': 0.90,
    },
]

print("=== SAVINGS OPPORTUNITIES ===\n")
print(f"Based on your January-February spending averages:\n")

total_monthly_savings = 0
total_annual_savings = 0

for rule in SAVINGS_RULES:
    cat = rule['category']
    if cat in monthly_by_cat.index and monthly_by_cat[cat] > 0:
        monthly_spend = monthly_by_cat[cat]
        monthly_save = monthly_spend * rule['save_pct']
        annual_save = monthly_save * 12
        total_monthly_savings += monthly_save
        total_annual_savings += annual_save
        
        print(f"  {cat}")
        print(f"    Current:    GBP {monthly_spend:>7,.2f}/month")
        print(f"    Suggestion: {rule['suggestion']}")
        print(f"    Potential:  GBP {monthly_save:>7,.2f}/month  |  GBP {annual_save:>8,.2f}/year")
        print()

print(f"  {'=' * 55}")
print(f"  TOTAL POTENTIAL SAVINGS")
print(f"    Monthly: GBP {total_monthly_savings:>7,.2f}")
print(f"    Annual:  GBP {total_annual_savings:>8,.2f}")
print(f"  {'=' * 55}")

=== SAVINGS OPPORTUNITIES ===

Based on your January-February spending averages:

  Food Delivery
    Current:    GBP   19.10/month
    Suggestion: Cook instead of ordering delivery once a week
    Potential:  GBP    9.55/month  |  GBP   114.60/year

  Eating Out
    Current:    GBP   83.26/month
    Suggestion: Meal prep lunches instead of eating out twice a week
    Potential:  GBP   24.98/month  |  GBP   299.74/year

  Coffee & Cafe
    Current:    GBP    5.30/month
    Suggestion: Make coffee at home 3 days a week
    Potential:  GBP    3.18/month  |  GBP    38.16/year

  Transport
    Current:    GBP  183.71/month
    Suggestion: Walk or cycle for short trips, use bus more
    Potential:  GBP   36.74/month  |  GBP   440.89/year

  Subscriptions
    Current:    GBP   11.70/month
    Suggestion: Review and cancel unused subscriptions
    Potential:  GBP    3.51/month  |  GBP    42.10/year

  Shopping
    Current:    GBP   63.73/month
    Suggestion: Implement a 48-hour rule before n

In [8]:
# Cell 7: Complete Financial Health Summary
# WHY: This pulls together every analytical module we have built
# into a single data structure. In the web app, this is what gets
# passed to the AI narrator and displayed on the dashboard.

spending = df[df['direction'] == 'OUT'].copy()
income = df[df['direction'] == 'IN'].copy()

# Monthly averages (Jan + Feb only)
spending['month'] = pd.to_datetime(spending['date_iso']).dt.to_period('M')
full_month_spending = spending[spending['month'].isin([pd.Period('2026-01'), pd.Period('2026-02')])]
avg_monthly_spend = full_month_spending['money_out'].sum() / 2
avg_monthly_income = income['money_in'].sum() / 2.5  # 2.5 months of data

print("=" * 60)
print("  SPENDSCOPE FINANCIAL HEALTH REPORT")
print("  Period: January 2026 - March 2026")
print("=" * 60)

print(f"\n  OVERVIEW")
print(f"  ─────────────────────────────────")
print(f"  Avg Monthly Income:    GBP {avg_monthly_income:>8,.2f}")
print(f"  Avg Monthly Spending:  GBP {avg_monthly_spend:>8,.2f}")
print(f"  Avg Monthly Surplus:   GBP {avg_monthly_income - avg_monthly_spend:>8,.2f}")

# Top 5 spending categories (lifestyle only)
lifestyle_cats = ['Transport', 'Groceries', 'Eating Out', 'Shopping', 
                  'Food Delivery', 'Coffee & Cafe', 'Subscriptions']
lifestyle = full_month_spending[full_month_spending['category'].isin(lifestyle_cats)]
lifestyle_monthly = lifestyle.groupby('category')['money_out'].sum() / 2

print(f"\n  TOP LIFESTYLE SPENDING (monthly avg)")
print(f"  ─────────────────────────────────")
for cat, amt in lifestyle_monthly.sort_values(ascending=False).items():
    bar = '█' * int(amt / 5)
    print(f"  {cat:<20} GBP {amt:>7,.2f}  {bar}")

# Subscriptions
print(f"\n  RECURRING SUBSCRIPTIONS")
print(f"  ─────────────────────────────────")
print(f"  Voxi Mobile           GBP  12.00/month")
print(f"  Apple Subscription    GBP   2.99/month")
print(f"  Google One            GBP   1.72/month")
print(f"  Canva (via Google)    GBP   0.99/month")
print(f"  Anthem Homes (Rent)   GBP 660.00/month")

# Key insights
print(f"\n  KEY INSIGHTS")
print(f"  ─────────────────────────────────")
print(f"  1. Transport is your #1 lifestyle expense at GBP {lifestyle_monthly.get('Transport', 0):,.2f}/month")
print(f"     Uber alone accounts for GBP {full_month_spending[full_month_spending['merchant'] == 'Uber']['money_out'].sum() / 2:,.2f}/month")
print(f"  2. Eating out increased 55% from January to February")
print(f"  3. Food delivery appeared for the first time in February")
print(f"  4. Bank fees of GBP {lifestyle_monthly.get('Bank Fees', full_month_spending[full_month_spending['category'] == 'Bank Fees']['money_out'].sum() / 2):,.2f}/month are avoidable with a fee-free card")
print(f"  5. Potential annual savings: GBP {total_annual_savings:,.2f}")

print(f"\n  ANOMALIES FLAGGED: {len(anomalies_df)}")
print(f"  ─────────────────────────────────")
for _, row in anomalies_df.head(5).iterrows():
    print(f"  {row['date']}  {row['merchant']:<25} GBP {row['amount']:>7,.2f}")

print(f"\n{'=' * 60}")
print(f"  Report generated by SpendScope")
print(f"{'=' * 60}")

  SPENDSCOPE FINANCIAL HEALTH REPORT
  Period: January 2026 - March 2026

  OVERVIEW
  ─────────────────────────────────
  Avg Monthly Income:    GBP 2,115.43
  Avg Monthly Spending:  GBP 2,273.76
  Avg Monthly Surplus:   GBP  -158.34

  TOP LIFESTYLE SPENDING (monthly avg)
  ─────────────────────────────────
  Transport            GBP  183.71  ████████████████████████████████████
  Groceries            GBP   91.06  ██████████████████
  Eating Out           GBP   83.26  ████████████████
  Shopping             GBP   63.73  ████████████
  Food Delivery        GBP   19.10  ███
  Subscriptions        GBP   11.70  ██
  Coffee & Cafe        GBP    5.30  █

  RECURRING SUBSCRIPTIONS
  ─────────────────────────────────
  Voxi Mobile           GBP  12.00/month
  Apple Subscription    GBP   2.99/month
  Google One            GBP   1.72/month
  Canva (via Google)    GBP   0.99/month
  Anthem Homes (Rent)   GBP 660.00/month

  KEY INSIGHTS
  ─────────────────────────────────
  1. Transport is your

In [4]:
print("Kernel is working")

Kernel is working


In [5]:
%pip install kagglehub


   ---------------------------------------- 0/4 [tqdm]
   ---------------------------------------- 0/4 [tqdm]
   ---------------------------------------- 0/4 [tqdm]
   ---------------------------------------- 0/4 [tqdm]
   ---------------------------------------- 0/4 [tqdm]
   ---------------------------------------- 0/4 [tqdm]
   ---------- ----------------------------- 1/4 [protobuf]
   ---------- ----------------------------- 1/4 [protobuf]
   ---------- ----------------------------- 1/4 [protobuf]
   ---------- ----------------------------- 1/4 [protobuf]
   ---------- ----------------------------- 1/4 [protobuf]
   ---------- ----------------------------- 1/4 [protobuf]
   ---------- ----------------------------- 1/4 [protobuf]
   ---------- ----------------------------- 1/4 [protobuf]
   ---------- ----------------------------- 1/4 [protobuf]
   ---------- ----------------------------- 1/4 [protobuf]
   ---------- ----------------------------- 1/4 [protobuf]
   ---------- ------


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import pandas as pd
from pathlib import Path
import os

PROJECT_ROOT = Path("..").resolve()
SYNTH = PROJECT_ROOT / "data" / "synthetic"

print("Files in data/synthetic:")
for f in SYNTH.iterdir():
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"  {f.name}  ({size_mb:.2f} MB)")

Files in data/synthetic:
  Banking_Transactions_USA_2023_2024.csv  (1.28 MB)
  demo_transactions.csv  (0.04 MB)


In [7]:
# Load and inspect the Kaggle dataset
kaggle_df = pd.read_csv(SYNTH / "Banking_Transactions_USA_2023_2024.csv")

print(f"Shape: {kaggle_df.shape[0]} rows x {kaggle_df.shape[1]} columns")
print(f"\nColumns: {list(kaggle_df.columns)}")
print(f"\nData types:\n{kaggle_df.dtypes}")
print(f"\nFirst 5 rows:")
kaggle_df.head()

Shape: 5389 rows x 20 columns

Columns: ['Transaction_ID', 'Account_Number', 'Transaction_Date', 'Transaction_Amount', 'Merchant_Name', 'Transaction_Type', 'Category', 'City', 'Country', 'Payment_Method', 'Customer_Age', 'Customer_Gender', 'Customer_Occupation', 'Customer_Income', 'Account_Balance', 'Transaction_Status', 'Fraud_Flag', 'Discount_Applied', 'Loyalty_Points_Earned', 'Transaction_Description']

Data types:
Transaction_ID                 str
Account_Number                 str
Transaction_Date               str
Transaction_Amount         float64
Merchant_Name                  str
Transaction_Type               str
Category                       str
City                           str
Country                        str
Payment_Method                 str
Customer_Age                 int64
Customer_Gender                str
Customer_Occupation            str
Customer_Income            float64
Account_Balance            float64
Transaction_Status             str
Fraud_Flag        

,Transaction_ID,Account_Number,Transaction_Date,Transaction_Amount,Merchant_Name,Transaction_Type,Category,City,Country,Payment_Method,Customer_Age,Customer_Gender,Customer_Occupation,Customer_Income,Account_Balance,Transaction_Status,Fraud_Flag,Discount_Applied,Loyalty_Points_Earned,Transaction_Description
0,bdd640fb-0667-4ad1-9c80-317fa3b1799d,IUPM04409079772781,2023-11-05 15:54:38,3198.94,Houston Group,Debit,Transport,Phoenix,USA,Online Transfer,55,Others,Quality manager,80466.03,350.28,Failed,No,True,304,Recently company detail form range a.
1,23b8c1e9-3924-46de-beb1-3b9046685257,BLAT22216107051843,2024-04-21 22:21:55,129.93,Anderson-Phillips,Credit,Grocery,Philadelphia,USA,Debit Card,26,Others,Civil Service fast streamer,145574.25,9797.81,Pending,Yes,False,383,Anything son baby power heart will not up.
2,bd9c66b3-ad3c-4d6d-9a3d-1fa7bc8960a9,UTXA55295806601382,2023-07-17 13:25:56,1378.77,Jensen Group,Credit,Shopping,New York,USA,Debit Card,29,Others,"Pilot, airline",33447.18,12399.85,Failed,Yes,False,497,Form world around green bar environment pattern.
3,972a8469-1641-4f82-8b9d-2434e465e150,XICF70493862044851,2023-06-27 16:09:52,1119.94,"Nelson, Gomez and Rodriguez",Credit,Healthcare,Dallas,USA,Online Transfer,60,Male,"Radiographer, therapeutic",108801.45,16057.64,Failed,Yes,True,495,Order evening source these opportunity trade i...
4,17fc695a-07a0-4a6e-8822-e8f36c031199,KOSW19711121259020,2024-03-26 23:45:31,3683.67,Caldwell Group,Credit,Entertainment,San Jose,USA,E-Wallet,29,Others,Diplomatic Services operational officer,100985.12,14940.54,Failed,Yes,True,292,Exactly politics door suggest.


In [8]:
# Transform Kaggle data into SpendScope format
# WHY: Our app expects a specific structure. This proves the
# intelligence layer works on any dataset, not just Lloyds.

print(f"Raw shape: {kaggle_df.shape}")
print(f"\nUnique categories: {kaggle_df['Category'].unique()}")
print(f"\nTransaction types: {kaggle_df['Transaction_Type'].unique()}")
print(f"\nTransaction status: {kaggle_df['Transaction_Status'].unique()}")
print(f"\nDate range: {kaggle_df['Transaction_Date'].min()} to {kaggle_df['Transaction_Date'].max()}")
print(f"\nAmount range: {kaggle_df['Transaction_Amount'].min()} to {kaggle_df['Transaction_Amount'].max()}")
print(f"\nNull counts:\n{kaggle_df.isna().sum()}")

Raw shape: (5389, 20)

Unique categories: <StringArray>
[    'Transport',       'Grocery',      'Shopping',    'Healthcare',
 'Entertainment',       'Savings',      'Clothing',       'Housing',
   'Electronics',     'Education',     'Utilities',       'Fitness',
        'Travel',          'Food']
Length: 14, dtype: str

Transaction types: <StringArray>
['Debit', 'Credit']
Length: 2, dtype: str

Transaction status: <StringArray>
['Failed', 'Pending', 'Success']
Length: 3, dtype: str

Date range: 2023-01-21 08:42:42 to 2025-01-20 12:21:18

Amount range: 5.46 to 4999.54

Null counts:
Transaction_ID             0
Account_Number             0
Transaction_Date           0
Transaction_Amount         0
Merchant_Name              0
Transaction_Type           0
Category                   0
City                       0
Country                    0
Payment_Method             0
Customer_Age               0
Customer_Gender            0
Customer_Occupation        0
Customer_Income            0
Accoun

In [9]:
# Transform into SpendScope format
# Only keep successful transactions (Failed/Pending are not real spending)
clean = kaggle_df[kaggle_df['Transaction_Status'] == 'Success'].copy()
print(f"After removing Failed/Pending: {len(clean)} transactions")

# Map to our format
spendscope_df = pd.DataFrame({
    'date_iso': pd.to_datetime(clean['Transaction_Date']).dt.strftime('%Y-%m-%d'),
    'description': clean['Transaction_Description'],
    'merchant': clean['Merchant_Name'],
    'category': clean['Category'],
    'type': clean['Transaction_Type'].map({'Debit': 'DEB', 'Credit': 'FPI'}),
    'money_in': clean.apply(lambda r: r['Transaction_Amount'] if r['Transaction_Type'] == 'Credit' else 0, axis=1),
    'money_out': clean.apply(lambda r: r['Transaction_Amount'] if r['Transaction_Type'] == 'Debit' else 0, axis=1),
    'balance': clean['Account_Balance'],
    'direction': clean['Transaction_Type'].map({'Debit': 'OUT', 'Credit': 'IN'}),
})

# Sort by date
spendscope_df = spendscope_df.sort_values('date_iso').reset_index(drop=True)

print(f"\nTransformed shape: {spendscope_df.shape}")
print(f"Date range: {spendscope_df['date_iso'].min()} to {spendscope_df['date_iso'].max()}")
print(f"Unique merchants: {spendscope_df['merchant'].nunique()}")
print(f"Categories: {spendscope_df['category'].nunique()}")

total_in = spendscope_df['money_in'].sum()
total_out = spendscope_df['money_out'].sum()
print(f"\nTotal In:  ${total_in:,.2f}")
print(f"Total Out: ${total_out:,.2f}")
print(f"Net:       ${total_in - total_out:,.2f}")

print(f"\nSample:")
spendscope_df.head()

After removing Failed/Pending: 1789 transactions

Transformed shape: (1789, 9)
Date range: 2023-01-22 to 2025-01-20
Unique merchants: 1714
Categories: 14

Total In:  $2,218,198.35
Total Out: $2,291,738.44
Net:       $-73,540.09

Sample:


,date_iso,description,merchant,category,type,money_in,money_out,balance,direction
0,2023-01-22,Evidence card realize.,Lynch LLC,Food,DEB,0.0,789.73,3410.98,OUT
1,2023-01-23,Laugh write catch dog.,Cole-Kirk,Transport,DEB,0.0,2165.93,13019.98,OUT
2,2023-01-23,Class economy pattern identify beautiful look ...,Ramos-Evans,Grocery,DEB,0.0,2502.77,11903.55,OUT
3,2023-01-23,Easy throw late our fire arrive.,"Welch, Rangel and Gallagher",Utilities,DEB,0.0,1388.45,2402.55,OUT
4,2023-01-23,Public probably mother bit.,Rose-Jackson,Clothing,DEB,0.0,1352.94,2621.79,OUT


In [10]:
# Save as frontend-ready JSON and update the API data
output_json = PROJECT_ROOT / "data" / "processed" / "transactions_frontend.json"
output_csv = PROJECT_ROOT / "data" / "processed" / "transactions_frontend.csv"

# Also update the demo data for the React frontend
demo_json = PROJECT_ROOT / "frontend" / "public" / "demo_data.json"

spendscope_df.to_json(output_json, orient='records', indent=2)
spendscope_df.to_csv(output_csv, index=False)
spendscope_df.to_json(demo_json, orient='records', indent=2)

print(f"Saved {len(spendscope_df)} transactions to:")
print(f"  {output_json}")
print(f"  {output_csv}")
print(f"  {demo_json}")
print(f"\nNow refresh localhost:5173 to see the app with 1,789 transactions!")

Saved 1789 transactions to:
  C:\Users\riyaw\OneDrive\Desktop\SpendScope\data\processed\transactions_frontend.json
  C:\Users\riyaw\OneDrive\Desktop\SpendScope\data\processed\transactions_frontend.csv
  C:\Users\riyaw\OneDrive\Desktop\SpendScope\frontend\public\demo_data.json

Now refresh localhost:5173 to see the app with 1,789 transactions!
